In [6]:
import sys
import logging
from typing import Dict, Any, List, Optional, Annotated
from typing_extensions import TypedDict
import uuid

# 导入 LangGraph 核心组件
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import AIMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool

logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("PolicyGateHITLHedged")

# ============================================================================
# 1. 具备幂等性与动态账本校验的 Mock 后端
# ============================================================================

class MockBankingBackend:
    def __init__(self, initial_balance: float = 5000.0):
        self.balance = initial_balance
        self.debit_count = 0
        self.ledger: Dict[str, dict] = {} # 幂等键 -> 执行结果

    def get_balance(self) -> float:
        return self.balance

    def execute_transfer(self, amount: float, recipient: str, idempotency_key: str) -> dict:
        # 【幂等防线】如果该幂等键已处理过，直接返回历史结果，绝不重复扣款！
        if idempotency_key in self.ledger:
            logger.info(f"🛡️ [Idempotency Protection] 捕获到重复键: {idempotency_key}，直接返回历史结果，不重复扣款。")
            return self.ledger[idempotency_key]

        # 实时余额校验
        if amount > self.balance:
            return {"success": False, "reason": "INSUFFICIENT_FUNDS"}

        self.balance -= amount
        self.debit_count += 1
        
        result = {
            "success": True, 
            "remaining_balance": self.balance, 
            "transferred": amount,
            "debit_count": self.debit_count
        }
        self.ledger[idempotency_key] = result
        return result

# 全局后端实例
backend = MockBankingBackend(initial_balance=5000.0)

@tool
def get_balance() -> dict:
    """查询当前账户余额"""
    logger.info("⚙️ [Tool Execution] 执行只读工具: get_balance()")
    return {"balance": backend.get_balance()}

@tool
def execute_transfer(amount: float, recipient: str, idempotency_key: str) -> dict:
    """执行高危副作用转账工具"""
    logger.warning(f"🚨 [Tool Execution] 执行转账: amount={amount}, recipient={recipient}, key={idempotency_key}")
    return backend.execute_transfer(amount, recipient, idempotency_key)

tools = [get_balance, execute_transfer]
tool_node_executor = ToolNode(tools)

# ============================================================================
# 2. 状态定义 (AgentState)
# ============================================================================

class PendingCallInfo(TypedDict):
    tool_call_id: str
    tool_name: str
    args: Dict[str, Any]

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    pending_call: Optional[PendingCallInfo]      # 当前等待审批的高危调用
    approval_result: Optional[Dict[str, Any]]    # 恢复后收到的审批数据
    filtered_ai_message: Optional[AIMessage]     # 供 ToolNode 专用的过滤消息

# ============================================================================
# 3. 图节点定义 (Nodes with strict revalidation & ToolNode routing)
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    """1. LLM 决策节点：生成复合 Tool Calls"""
    logger.info("🤖 [agent_node] LLM 正在生成 Tool Calls...")
    ai_msg = AIMessage(
        content="请帮我查询余额并转账 1000 元给 Alice。",
        tool_calls=[
            {"id": "call_bal_01", "name": "get_balance", "args": {}},
            {"id": "call_trans_01", "name": "execute_transfer", "args": {
                "amount": 1000.0, 
                "recipient": "Alice", 
                "idempotency_key": "txn-key-9988"
            }}
        ]
    )
    return {"messages": [ai_msg]}


def policy_gate_node(state: AgentState) -> AgentState:
    """2. Policy Gate 节点：只读工具直接放行，副作用工具暂存为 pending_call"""
    logger.info("🛡️ [policy_gate_node] 审查 Tool Calls...")
    last_message = [m for m in state["messages"] if isinstance(m, AIMessage)][-1]
    
    allowed_calls = []
    pending = None
    
    for tc in last_message.tool_calls:
        if tc["name"] == "get_balance":
            allowed_calls.append(tc)
        elif tc["name"] == "execute_transfer":
            pending = {
                "tool_call_id": tc["id"],
                "tool_name": tc["name"],
                "args": tc["args"]
            }
            
    filtered_ai = AIMessage(content=last_message.content, tool_calls=allowed_calls)
    return {
        "filtered_ai_message": filtered_ai,
        "pending_call": pending
    }


def approval_node(state: AgentState) -> AgentState:
    """
    3. 审批与中断节点 (Human-in-the-Loop)：
    调用内置 interrupt() 暂停图的执行。
    
    【架构说明 - 审批身份与凭证验证】:
    外部宿主/Webhook 在接收到 interrupt 载荷并调用 Command(resume=...) 前，
    必须通过 OAuth2 / Session 验证操作者的身份、RBAC 权限及审批有效期。
    图内部的 revalidate 节点则专注于防参数篡改及业务上下文的一致性核对。
    """
    logger.info("⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...")
    p_call = state["pending_call"]
    
    approval_data = interrupt({
        "tool_call_id": p_call["tool_call_id"],
        "tool_name": p_call["tool_name"],
        "args": p_call["args"],
        "question": "是否批准这笔高危转账？"
    })
    
    logger.info(f"▶️ [approval_node] 收到恢复数据 (resume): {approval_data}")
    return {"approval_result": approval_data}


def revalidate_node(state: AgentState) -> AgentState:
    """
    4. 严格的恢复后重新校验节点：
    使用显式条件分支进行安全拦截，严禁盲目信任外部 resume 载荷。
    """
    logger.info("🔍 [revalidate_node] 收到恢复信号，开始严格的安全重新校验...")
    approval = state.get("approval_result", {})
    p_call = state.get("pending_call")
    
    # 分支 1：检查是否明确批准
    if not approval.get("approved", False):
        logger.warning("🛑 [Revalidation Denied] 用户拒绝了该笔审批，流程终止。")
        return {"messages": [ToolMessage(content="ERROR: USER_REJECTED_APPROVAL", tool_call_id=p_call["tool_call_id"])]}

    # 分支 2：精确核对 tool_call_id、tool_name 与 args 参数（防止参数篡改与挂羊头卖狗肉）
    if approval.get("tool_call_id") != p_call["tool_call_id"] or \
       approval.get("tool_name") != p_call["tool_name"] or \
       approval.get("args") != p_call["args"]:
        logger.warning("🛑 [Revalidation Mismatch] 审批对象或参数与待执行调用不匹配（检测到参数篡改），安全拦截！")
        return {"messages": [ToolMessage(content="ERROR: APPROVAL_PARAMETER_MISMATCH", tool_call_id=p_call["tool_call_id"])]}

    # 分支 3：暂停后实时重新检查余额
    current_balance = backend.get_balance()
    required_amount = p_call["args"]["amount"]
    if required_amount > current_balance:
        logger.warning(f"🛑 [Revalidation Insufficient Funds] 暂停期间余额降至 {current_balance}，不足以支付 {required_amount}，取消转账！")
        return {"messages": [ToolMessage(content="ERROR: CANCELLED_DUE_TO_BALANCE_DROPPED_DURING_PAUSE", tool_call_id=p_call["tool_call_id"])]}

    # 校验全部通过，构造合规的 AIMessage，准备交由标准 ToolNode 执行
    logger.info("✅ [Revalidation Passed] 重新校验全部通过，构建受信任的 Tool Call...")
    authorized_ai = AIMessage(content="Authorized execution", tool_calls=[{
        "id": p_call["tool_call_id"],
        "name": p_call["tool_name"],
        "args": p_call["args"]
    }])
    
    return {"filtered_ai_message": authorized_ai}


def tool_execution_router(state: AgentState) -> str:
    """条件路由 1：判断是否有只读工具需要先执行"""
    filtered_ai = state.get("filtered_ai_message")
    if filtered_ai and len(filtered_ai.tool_calls) > 0 and not state.get("pending_call"):
        return "run_readonly_tools"
    return "go_to_approval"


def post_revalidation_router(state: AgentState) -> str:
    """条件路由 2：根据重新校验结果决定进入标准 ToolNode 还是跳过/拒绝"""
    last_msg = state["messages"][-1] if state["messages"] else None
    # 如果重新校验失败生成了错误 ToolMessage，则直接结束
    if isinstance(last_msg, ToolMessage) and "ERROR" in str(last_msg.content):
        return "skip_tools"
    return "execute_via_tool_node"


def readonly_tool_executor_node(state: AgentState) -> AgentState:
    """执行只读工具（如 get_balance）"""
    logger.info("⚡ [readonly_tool_executor_node] 执行只读工具...")
    filtered_ai = state["filtered_ai_message"]
    execution_state = {"messages": [filtered_ai]}
    result = tool_node_executor.invoke(execution_state)
    return {"messages": result["messages"]}


def authorized_tool_node_executor(state: AgentState) -> AgentState:
    """
    【核心架构对齐】：获批的转账必须且只能通过标准的 ToolNode 执行，回归安全边界
    """
    logger.info("⚡ [authorized_tool_node_executor] 经安全网关放行，交由标准 ToolNode 执行副作用工具...")
    filtered_ai = state["filtered_ai_message"]
    execution_state = {"messages": [filtered_ai]}
    result = tool_node_executor.invoke(execution_state)
    return {"messages": result["messages"]}


def skip_execution_node(state: AgentState) -> AgentState:
    """未通过校验时的兜底空节点"""
    return {}

# ============================================================================
# 4. 构建带 Checkpointer 的状态图
# ============================================================================

def build_hitl_graph():
    builder = StateGraph(AgentState)
    
    builder.add_node("agent", agent_node)
    builder.add_node("policy_gate", policy_gate_node)
    builder.add_node("readonly_tool_node", readonly_tool_executor_node)
    builder.add_node("approval_node", approval_node)
    builder.add_node("revalidate_node", revalidate_node)
    builder.add_node("tool_node", authorized_tool_node_executor)
    builder.add_node("skip_node", skip_execution_node)
    
    # 边与条件路由
    builder.add_edge(START, "agent")
    builder.add_edge("agent", "policy_gate")
    
    builder.add_conditional_edges(
        "policy_gate",
        tool_execution_router,
        {
            "run_readonly_tools": "readonly_tool_node",
            "go_to_approval": "approval_node"
        }
    )
    
    builder.add_edge("readonly_tool_node", "approval_node")
    builder.add_edge("approval_node", "revalidate_node")
    
    builder.add_conditional_edges(
        "revalidate_node",
        post_revalidation_router,
        {
            "execute_via_tool_node": "tool_node",
            "skip_tools": "skip_node"
        }
    )
    
    builder.add_edge("tool_node", END)
    builder.add_edge("skip_node", END)
    
    return builder.compile(checkpointer=InMemorySaver())

# ============================================================================
# 5. 测试用例与验收验证 (含补充的拒绝与篡改测试断言)
# ============================================================================

def run_day38_revision_tests():
    print("=" * 70)
    print("🧪 Day 38 修订版: HITL 安全边界与参数篡改防线验收测试")
    print("=" * 70)

    # ------------------------------------------------------------------------
    # 场景 1：用户明确拒绝审批 (approved=False) -> 验证扣款 0 次
    # ------------------------------------------------------------------------
    print("\n--- [Test 1] 用户明确拒绝审批 ---")
    global backend
    backend = MockBankingBackend(initial_balance=5000.0)
    graph = build_hitl_graph()
    thread_config_1 = {"configurable": {"thread_id": "thread-reject-01"}}
    
    # 运行至暂停
    graph.invoke({"messages": []}, config=thread_config_1)
    
    # 用户拒绝恢复
    reject_command = Command(resume={"approved": False})
    graph.invoke(reject_command, config=thread_config_1)
    
    print(f"拒绝后的最终余额: {backend.balance}, 扣款次数: {backend.debit_count}")
    assert backend.debit_count == 0, "用户拒绝后扣款必须为 0"
    print("✅ 测试 1 通过：用户拒绝审批，扣款 0 次。")

    # ------------------------------------------------------------------------
    # 场景 2：审批参数被篡改 (批准 1000 给 Alice，但恢复载荷尝试修改参数) -> 验证扣款 0 次
    # ------------------------------------------------------------------------
    print("\n--- [Test 2] 审批参数被篡改拦截 ---")
    backend = MockBankingBackend(initial_balance=5000.0)
    thread_config_2 = {"configurable": {"thread_id": "thread-tamper-01"}}
    
    # 运行至暂停（原始待执行参数为 amount: 1000.0, recipient: "Alice"）
    graph.invoke({"messages": []}, config=thread_config_2)
    
    # 恶意/篡改的恢复载荷：尝试偷偷把金额改成 2000.0 或者换收款人
    tampered_command = Command(resume={
        "approved": True,
        "tool_call_id": "call_trans_01",
        "tool_name": "execute_transfer",
        "args": {"amount": 2000.0, "recipient": "Bob", "idempotency_key": "txn-key-9988"}
    })
    graph.invoke(tampered_command, config=thread_config_2)
    
    print(f"参数篡改拦截后的最终余额: {backend.balance}, 扣款次数: {backend.debit_count}")
    assert backend.debit_count == 0, "参数被篡改后必须拦截，扣款必须为 0"
    print("✅ 测试 2 通过：成功拦截参数篡改，扣款 0 次。")

    # ------------------------------------------------------------------------
    # 场景 3：正常批准且参数一致，通过 ToolNode 成功转账并验证幂等
    # ------------------------------------------------------------------------
    print("\n--- [Test 3] 正常批准，通过 ToolNode 转账并验证幂等防重 ---")
    backend = MockBankingBackend(initial_balance=5000.0)
    thread_config_3 = {"configurable": {"thread_id": "thread-success-01"}}
    
    # 1. 运行至暂停
    graph.invoke({"messages": []}, config=thread_config_3)
    
    target_key = "txn-key-9988"
    
    # 2. 合规且参数完全一致的恢复载荷
    valid_command = Command(resume={
        "approved": True,
        "tool_call_id": "call_trans_01",
        "tool_name": "execute_transfer",
        "args": {"amount": 1000.0, "recipient": "Alice", "idempotency_key": target_key}
    })
    graph.invoke(valid_command, config=thread_config_3)
    
    print(f"正常转账成功后的余额: {backend.balance}, 扣款次数: {backend.debit_count}")
    assert backend.debit_count == 1, "应当通过 ToolNode 成功扣款 1 次"

    # 3. 模拟使用相同的 Idempotency Key 再次尝试执行（测试幂等保护）
    print(">>> 模拟相同幂等键再次尝试重复执行...")
    backend.execute_transfer(amount=1000.0, recipient="Alice", idempotency_key=target_key)
    
    print(f"再次执行后的最终余额: {backend.balance}, 最终扣款次数: {backend.debit_count}")
    assert backend.debit_count == 1, "幂等键生效，重复执行绝不二次扣款！"
    
    print("\n🎉 Day 38 修订版所有验收测试全部完美通过：严格参数核对、ToolNode 边界闭环、拒绝与篡改防御全部达标！")

if __name__ == "__main__":
    run_day38_revision_tests()

2026-10-20 16:42:02,019 - [INFO] - 🤖 [agent_node] LLM 正在生成 Tool Calls...
2026-10-20 16:42:02,021 - [INFO] - 🛡️ [policy_gate_node] 审查 Tool Calls...
2026-10-20 16:42:02,023 - [INFO] - ⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...
2026-10-20 16:42:02,027 - [INFO] - ⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...
2026-10-20 16:42:02,028 - [INFO] - ▶️ [approval_node] 收到恢复数据 (resume): {'approved': False}
2026-10-20 16:42:02,031 - [INFO] - 🔍 [revalidate_node] 收到恢复信号，开始严格的安全重新校验...
2026-10-20 16:42:02,032 - [WARNING] - 🛑 [Revalidation Denied] 用户拒绝了该笔审批，流程终止。
2026-10-20 16:42:02,039 - [INFO] - 🤖 [agent_node] LLM 正在生成 Tool Calls...
2026-10-20 16:42:02,041 - [INFO] - 🛡️ [policy_gate_node] 审查 Tool Calls...
2026-10-20 16:42:02,043 - [INFO] - ⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...
2026-10-20 16:42:02,048 - [INFO] - ⏸️ [approval_node] 触发 interrupt()，等待人类审批输入...
2026-10-20 16:42:02,049 - [INFO] - ▶️ [approval_node] 收到恢复数据 (resume): {'approved': True, 'tool_call_id': 'call_trans_01', 'tool_name': 

🧪 Day 38 修订版: HITL 安全边界与参数篡改防线验收测试

--- [Test 1] 用户明确拒绝审批 ---
拒绝后的最终余额: 5000.0, 扣款次数: 0
✅ 测试 1 通过：用户拒绝审批，扣款 0 次。

--- [Test 2] 审批参数被篡改拦截 ---
参数篡改拦截后的最终余额: 5000.0, 扣款次数: 0
✅ 测试 2 通过：成功拦截参数篡改，扣款 0 次。

--- [Test 3] 正常批准，通过 ToolNode 转账并验证幂等防重 ---
正常转账成功后的余额: 4000.0, 扣款次数: 1
>>> 模拟相同幂等键再次尝试重复执行...
再次执行后的最终余额: 4000.0, 最终扣款次数: 1

🎉 Day 38 修订版所有验收测试全部完美通过：严格参数核对、ToolNode 边界闭环、拒绝与篡改防御全部达标！
